# PPIDB Test Notebook

This notebook tests the core functionality of the ppidb toolkit using the examples provided in the README.

## 1. Quick Start: Loading Data

In [2]:
from ppidb import PPIDataset

# Load the full database
ds = PPIDataset.load("ppidb_interaction.parquet")
# Inspect
ds.describe()

  PPIDataset Summary
  Pairs:       7,669,733
  Proteins:       99,120
  Sources:            39

  Top species:
    Human                      6,620,317
    S. cerevisiae                605,470
    Mouse                        113,472
    D. melanogaster               81,243
    A. thaliana                   70,653

  Top databases:
    ConsensusPathDB            5,260,435
    BioGRID                    1,899,417
    IID                        1,668,149
    HIPPIE                       758,853
    IntAct                       678,561

  Throughput type:
    LTP                   1,251,444
    negative_sample           2,888
    both                    283,886
    no_exp                5,186,433
    HTP                     945,082

  Interaction type:
    negative                  2,888
    positive              7,666,845


## 2. Filtering

In [3]:
# Filter by species (Human)
human = ds.filter.by_species("9606")
print(f"Human interactions: {len(human)}")

# Chained filtering
subset = (
    ds.filter.by_species("9606")
      .filter.positives_only()
      .filter.by_throughput("LTP")
      .filter.by_min_sources(2)
      .filter.by_database("BioGRID")
)
print(f"Filtered subset size: {len(subset)}")

Human interactions: 6588390
Filtered subset size: 539254


## 3. Sequence Retrieval (Local)

In [4]:
from ppidb.sequence import SequenceFetcher

fetcher = SequenceFetcher()
# Load sequences from local parquet file to avoid web fetching
fetcher.load_local_parquet("ppidb_protein.parquet")

# Fetch sequences for a small subset of proteins
sample_proteins = subset.proteins()[:5]
seqs = fetcher.fetch(sample_proteins, as_dict=True)

print(f"Fetched {len(seqs)} sequences.")
for pid, seq in seqs.items():
    print(f"{pid}: {seq[:20]}...")

Loaded 99,197 sequences from ppidb_protein.parquet
Fetching sequences: 5 total, 5 cached, 0 to download
Total sequences retrieved: 5 / 5
Fetched 5 sequences.
A0A024R161: MGAPLLSPGWGAGAAGRRWW...
A0A024RBG1: MMKFKPNQTRTYDREGFKKR...
A0A075B6E9: MQCLEMTTKRKIIGRLVPCR...
A0A087WU62: MAAPIPQGFSCLSRFLGWWS...
A0A087WUL8: MVVSAGPWSSEKAEMNILEI...


## 4. Negative Sampling

In [5]:
from ppidb.negative import NegativeSampler

# Use a smaller sample for quick testing
positives = ds.filter.high_confidence().sample(1000, seed=42)
sampler = NegativeSampler(positives)

# Strategy: Random negatives
negs = sampler.random_sample(ratio=1.0, seed=42)
labeled = NegativeSampler.combine(positives, negs, shuffle=True)

print(f"Labeled dataset size: {len(labeled)}")
labeled.describe()

Sampling 1,000 random negatives from 1,609 proteins...
Generated 1,000 random negative pairs.
Combined dataset: 1,000 positives + 1,000 negatives = 2,000 total (ratio 1.00:1)
Labeled dataset size: 2000
  PPIDataset Summary
  Pairs:           2,000
  Proteins:        1,609
  Sources:            32

  Top species:
    Human                          1,000
    Unknown                        1,000

  Top databases:
    random_negative                1,000
    IID                              961
    BioGRID                          808
    ConsensusPathDB                  671
    HIPPIE                           656

  Throughput type:
    negative_sample           1,000
    LTP                       1,000

  Interaction type:
    positive                  1,000
    negative                  1,000


## 5. Splitting (Greedy C3) & Leakage Evaluation

In [6]:
from ppidb.split import Splitter
from ppidb.split.c1c2c3 import compute_c1c2c3_stats

# Ensure we have sequences for the proteins in our labeled sample
all_proteins = labeled.proteins()
seqs = fetcher.fetch(all_proteins, as_dict=True)

splitter = Splitter(labeled)

# Using Greedy C3 split
split = splitter.greedy_c3_split(
    test_protein_frac=0.2, 
    seed=42,
    sequence_dict=seqs,
    identity_threshold=0.3
)

print(f"Train size: {len(split.train)}")
print(f"Test size: {len(split.test)}")

# Verify leakage
stats = compute_c1c2c3_stats(split.train, split.test)
print(stats)

Fetching sequences: 1,609 total, 1,609 cached, 0 to download
Total sequences retrieved: 1,609 / 1,609


d:\ppidb-master\ppidb\split\_clustering.py:28: UserWarning: MMseqs2 not found. Falling back to greedy pairwise clustering (slow for large datasets). Install MMseqs2 for better performance.
  warnings.warn(


Clustering 1609 sequences (greedy strategy)...


Clustering: 100%|██████████| 1609/1609 [00:12<00:00, 126.62it/s]

Train size: 1098
Test size: 432
C1/C2/C3 Evaluation Statistics
  C1 (Both-Seen):           0    0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  C2 (One-Seen):            0    0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  C3 (Neither-Seen):      432  100.0%  ██████████████████████████████
  Total:                  432

  Train proteins:       1,179
  Test proteins:          321
  Overlap proteins:         0



d:\ppidb-master\ppidb\split\c1c2c3.py:668: UserWarning: Strict C3: Removed 15 proteins from train_pool because they share >0.3 identity with test proteins.
  warnings.warn(
d:\ppidb-master\ppidb\split\c1c2c3.py:699: UserWarning: greedy_c3_split discarded 470 cross-pool pairs (23.5% of total).
  warnings.warn(


## 6. Full ML Workflow Integration Test

In [7]:
from ppidb import PPIDataset
from ppidb.split import Splitter
from ppidb.split.c1c2c3 import compute_c1c2c3_stats, split_test_by_c1c2c3
from ppidb.negative import NegativeSampler
from ppidb.sequence import SequenceFetcher

print("Starting full workflow test...")

# 1. Load and filter (taking a small sample for quick testing)
ds = PPIDataset.load("ppidb_interaction.parquet")
positives = ds.filter.high_confidence().sample(2000, seed=42)

# 2. Generate negatives
negs = NegativeSampler(positives).random_sample(ratio=1.0, seed=42)
labeled = NegativeSampler.combine(positives, negs)

# 3. Load sequences
fetcher = SequenceFetcher()
fetcher.load_local_parquet("ppidb_protein.parquet")
all_proteins = labeled.proteins()
seqs = fetcher.fetch(all_proteins, as_dict=True)

# 4. Split (C3-maximizing)
split = Splitter(labeled).greedy_c3_split(
    test_protein_frac=0.2, 
    seed=42, 
    sequence_dict=seqs,
    identity_threshold=0.3
)

# 5. Verify no leakage
stats = compute_c1c2c3_stats(split.train, split.test)
print(stats)

# 6. Fetch sequences for model input
train_seqs = fetcher.fetch(split.train.proteins(), as_dict=True)
test_seqs  = fetcher.fetch(split.test.proteins(),  as_dict=True)

print(f"Workflow completed. Train sequences: {len(train_seqs)}, Test sequences: {len(test_seqs)}")

Starting full workflow test...
Sampling 2,000 random negatives from 2,759 proteins...
Generated 2,000 random negative pairs.
Combined dataset: 2,000 positives + 2,000 negatives = 4,000 total (ratio 1.00:1)
Loaded 99,197 sequences from ppidb_protein.parquet
Fetching sequences: 2,759 total, 2,759 cached, 0 to download
Total sequences retrieved: 2,759 / 2,759
Clustering 2759 sequences (greedy strategy)...


Clustering: 100%|██████████| 2759/2759 [00:38<00:00, 71.23it/s]

C1/C2/C3 Evaluation Statistics
  C1 (Both-Seen):           0    0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  C2 (One-Seen):            0    0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  C3 (Neither-Seen):      852  100.0%  ██████████████████████████████
  Total:                  852

  Train proteins:       2,041
  Test proteins:          551
  Overlap proteins:         0
Fetching sequences: 2,041 total, 2,041 cached, 0 to download
Total sequences retrieved: 2,041 / 2,041
Fetching sequences: 551 total, 551 cached, 0 to download
Total sequences retrieved: 551 / 551
Workflow completed. Train sequences: 2041, Test sequences: 551



d:\ppidb-master\ppidb\split\c1c2c3.py:668: UserWarning: Strict C3: Removed 51 proteins from train_pool because they share >0.3 identity with test proteins.
  warnings.warn(
d:\ppidb-master\ppidb\split\c1c2c3.py:699: UserWarning: greedy_c3_split discarded 961 cross-pool pairs (24.0% of total).
  warnings.warn(
